In [2]:
# Cell 1: INSTALL + IMPORTS
!pip install opencv-python torch torchvision pillow numpy --quiet
import cv2
import urllib.request
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import subprocess
import time
import threading
print("✅ Alle imports geladen!")

✅ Alle imports geladen!


In [3]:
import cv2
import urllib.request
import os
import sys
import time
import threading
import numpy as np
from PIL import Image
import torch
import torch.nn as nn

# Optioneel geluid, maar we vallen terug zonder crash
try:
    import winsound
except Exception:
    winsound = None

# ===== DEVICE =====
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ===== MODEL =====
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2)  # 0=closed, 1=open
        )

    def forward(self, x):
        return self.model(x)

model = SimpleCNN().to(device)
model.load_state_dict(torch.load('fatigue_model.pth', map_location=device))
model.eval()
print("✅ Model geladen!")

# ===== FACE CASCADE =====
cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
cascade_path = "haarcascade_frontalface_default.xml"
if not os.path.exists(cascade_path):
    urllib.request.urlretrieve(cascade_url, cascade_path)

face_cascade = cv2.CascadeClassifier(cascade_path)

# ===== INSTELLINGEN =====
CONF_THRESHOLD = 0.70
CLOSED_ALARM_SECONDS = 2.0
SOFT_WARN_SECONDS = 0.8
MIN_FACE_SIZE = (60, 60)

# ===== STATE =====
blink_count = 0
eyes_closed_since = None
was_closed_prev = False
alarm_active = False
last_soft_warn = 0.0
last_loud_alarm = 0.0

# ===== GELUID =====
def play_soft_beep():
    if winsound:
        winsound.MessageBeep(winsound.MB_ICONASTERISK)
    else:
        print("\a", end="", flush=True)

def play_loud_alarm():
    if winsound:
        for _ in range(8):
            winsound.Beep(1400, 200)
            time.sleep(0.05)
    else:
        for _ in range(12):
            print("\a", end="", flush=True)
            time.sleep(0.15)

def start_soft_beep():
    threading.Thread(target=play_soft_beep, daemon=True).start()

def start_loud_alarm():
    threading.Thread(target=play_loud_alarm, daemon=True).start()

# ===== CAMERA =====
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 900)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 700)
print("🚨 Start | q = quit | s = alarm stop")

fps_t0 = time.time()
fps_n = 0
fps = 0.0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    fps_n += 1
    if time.time() - fps_t0 >= 1.0:
        fps = fps_n / (time.time() - fps_t0)
        fps_t0 = time.time()
        fps_n = 0

    now = time.time()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4, minSize=MIN_FACE_SIZE)

    is_closed_now = False
    conf = 0.0

    if len(faces) > 0:
        areas = [w * h for x, y, w, h in faces]
        idx = int(np.argmax(areas))
        x, y, w, h = faces[idx]

        face_crop = frame[y:y+h, x:x+w]
        if face_crop.size > 0:
            face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(face_rgb).resize((224, 224))
            img_array = np.array(img, dtype=np.float32) / 255.0
            img_tensor = torch.tensor(img_array).permute(2, 0, 1).unsqueeze(0).to(device)

            with torch.no_grad():
                logits = model(img_tensor)
                probs = torch.softmax(logits, 1)
                pred = int(torch.argmax(probs, 1).item())
                conf = float(probs.max().item())

            is_closed_now = (pred == 0 and conf >= CONF_THRESHOLD)

            color = (0, 0, 255) if is_closed_now else (0, 255, 0)
            label = "CLOSED" if is_closed_now else "OPEN"
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 4)
            cv2.putText(frame, f"{label} {conf:.0%}", (x, y-12),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

    # Blink tellen: closed -> open overgang
    if was_closed_prev and not is_closed_now:
        blink_count += 1

    # Closed-timer beheren
    if is_closed_now:
        if eyes_closed_since is None:
            eyes_closed_since = now
    else:
        eyes_closed_since = None

    was_closed_prev = is_closed_now

    closed_duration = 0.0 if eyes_closed_since is None else (now - eyes_closed_since)

    # ===== KEYS =====
    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        alarm_active = False
        print("🔇 Alarm gestopt")
    elif key == ord('q'):
        break

    # ===== STATUS / UI =====
    overlay = frame.copy()
    cv2.rectangle(overlay, (15, 15), (470, 170), (25, 25, 25), -1)
    frame = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)

    if closed_duration >= CLOSED_ALARM_SECONDS:
        alarm_active = True
        flash = (0, 0, 255) if int(now * 6) % 2 == 0 else (255, 255, 255)
        cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), flash, 12)
        cv2.putText(frame, "🚨 WAKE UP!", (250, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 0, 255), 5)

        if now - last_loud_alarm >= 2.5:
            last_loud_alarm = now
            start_loud_alarm()

    elif closed_duration >= SOFT_WARN_SECONDS or blink_count >= 12:
        alarm_active = False
        cv2.putText(frame, "😴 Moeheid: pauze aangeraden", (30, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)

        if now - last_soft_warn >= 5.0:
            last_soft_warn = now
            start_soft_beep()
    else:
        alarm_active = False
        cv2.putText(frame, "✅ Alert", (30, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)

    cv2.putText(frame, f"Blinks: {blink_count}", (30, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (230, 230, 230), 2)
    cv2.putText(frame, f"Eyes closed: {closed_duration:.1f}s", (30, 125),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (230, 230, 230), 2)
    cv2.putText(frame, f"FPS: {fps:.1f}", (30, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 200, 200), 2)
    cv2.putText(frame, "q = quit | s = stop alarm", (30, frame.shape[0]-20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (220, 220, 220), 2)

    cv2.imshow("Fatigue Monitor", frame)

cap.release()
cv2.destroyAllWindows()
print("✅ Gestopt")

Using device: cpu
✅ Model geladen!
🚨 Start | q = quit | s = alarm stop
🔇 Alarm gestopt
✅ Gestopt


In [4]:
import cv2
import urllib.request
import os
import time
import threading
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import simpleaudio as sa

# ===== DEVICE =====
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ===== MODEL =====
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.model(x)

model = SimpleCNN().to(device)
model.load_state_dict(torch.load('fatigue_model.pth', map_location=device))
model.eval()
print("✅ Model geladen!")

# ===== FACE CASCADE =====
cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
cascade_path = "haarcascade_frontalface_default.xml"
if not os.path.exists(cascade_path):
    urllib.request.urlretrieve(cascade_url, cascade_path)
face_cascade = cv2.CascadeClassifier(cascade_path)

# ===== SOUNDS =====
TIRED_SOUND = "mixkit-classic-alarm-995.wav"
SLEEP_SOUND = "mixkit-alert-alarm-1005.wav"

# ===== SETTINGS =====
CONF_THRESHOLD = 0.70
SOFT_WARN_SECONDS = 0.8
CLOSED_ALARM_SECONDS = 2.0
MIN_FACE_SIZE = (60, 60)

# ===== STATE =====
blink_count = 0
eyes_closed_since = None
was_closed_prev = False
last_soft_warn = 0.0
last_loud_alarm = 0.0
alarm_active = False

def play_for_1s(path):
    if not os.path.exists(path):
        print(f"Sound file not found: {path}")
        return
    wave = sa.WaveObject.from_wave_file(path)
    play_obj = wave.play()
    time.sleep(1.0)
    play_obj.stop()

def start_soft_sound():
    threading.Thread(target=play_for_1s, args=(TIRED_SOUND,), daemon=True).start()

def start_loud_sound():
    threading.Thread(target=play_for_1s, args=(SLEEP_SOUND,), daemon=True).start()

# ===== CAMERA =====
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 900)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 700)
print("🚨 Start | q = quit | s = alarm stop")

fps_t0 = time.time()
fps_n = 0
fps = 0.0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    fps_n += 1
    if time.time() - fps_t0 >= 1.0:
        fps = fps_n / (time.time() - fps_t0)
        fps_t0 = time.time()
        fps_n = 0

    now = time.time()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4, minSize=MIN_FACE_SIZE)

    is_closed_now = False
    conf = 0.0

    if len(faces) > 0:
        areas = [w * h for x, y, w, h in faces]
        idx = int(np.argmax(areas))
        x, y, w, h = faces[idx]

        face_crop = frame[y:y+h, x:x+w]
        if face_crop.size > 0:
            face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(face_rgb).resize((224, 224))
            img_array = np.array(img, dtype=np.float32) / 255.0
            img_tensor = torch.tensor(img_array).permute(2, 0, 1).unsqueeze(0).to(device)

            with torch.no_grad():
                logits = model(img_tensor)
                probs = torch.softmax(logits, 1)
                pred = int(torch.argmax(probs, 1).item())
                conf = float(probs.max().item())

            is_closed_now = (pred == 0 and conf >= CONF_THRESHOLD)

            color = (0, 0, 255) if is_closed_now else (0, 255, 0)
            label = "CLOSED" if is_closed_now else "OPEN"
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 4)
            cv2.putText(frame, f"{label} {conf:.0%}", (x, y-12),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

    if was_closed_prev and not is_closed_now:
        blink_count += 1

    if is_closed_now:
        if eyes_closed_since is None:
            eyes_closed_since = now
    else:
        eyes_closed_since = None

    was_closed_prev = is_closed_now
    closed_duration = 0.0 if eyes_closed_since is None else (now - eyes_closed_since)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        alarm_active = False
        print("🔇 Alarm gestopt")
    elif key == ord('q'):
        break

    overlay = frame.copy()
    cv2.rectangle(overlay, (15, 15), (470, 170), (25, 25, 25), -1)
    frame = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)

    if closed_duration >= CLOSED_ALARM_SECONDS:
        alarm_active = True
        flash = (0, 0, 255) if int(now * 6) % 2 == 0 else (255, 255, 255)
        cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), flash, 12)
        cv2.putText(frame, "🚨 WAKE UP!", (250, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 0, 255), 5)

        if now - last_loud_alarm >= 2.5:
            last_loud_alarm = now
            start_loud_sound()

    elif closed_duration >= SOFT_WARN_SECONDS or blink_count >= 12:
        alarm_active = False
        cv2.putText(frame, "😴 Moeheid: pauze aangeraden", (30, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)

        if now - last_soft_warn >= 5.0:
            last_soft_warn = now
            start_soft_sound()
    else:
        alarm_active = False
        cv2.putText(frame, "✅ Alert", (30, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)

    cv2.putText(frame, f"Blinks: {blink_count}", (30, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (230, 230, 230), 2)
    cv2.putText(frame, f"Eyes closed: {closed_duration:.1f}s", (30, 125),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (230, 230, 230), 2)
    cv2.putText(frame, f"FPS: {fps:.1f}", (30, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 200, 200), 2)
    cv2.putText(frame, "q = quit | s = stop alarm", (30, frame.shape[0]-20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (220, 220, 220), 2)

    cv2.imshow("Fatigue Monitor", frame)

cap.release()
cv2.destroyAllWindows()
print("✅ Gestopt")

Using device: cpu
✅ Model geladen!
🚨 Start | q = quit | s = alarm stop
✅ Gestopt
